# Recommendation System

In [1]:
import os,sys
print(os.getcwd())
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print("Project root directory:", project_root)
sys.path.append(project_root)

/home/zhaolei/meal-rec-ai/preprocess
Project root directory: /home/zhaolei/meal-rec-ai


In [2]:
# 加载数据
import pandas as pd
import warnings
from utils.data import root_path
from algo.classification import preprocess_data # 预处理


positive_samples = pd.read_csv(f'{root_path}daily_meal_plan_positive.csv')
negative_samples = pd.read_csv(f'{root_path}daily_meal_plan_negative.csv')
data = preprocess_data(positive_samples, negative_samples)

基于体征和疾病的膳食推荐算法
用户特征数: 23

数据预处理

原始数据统计:
  总记录数: 259958
  正例: 226758
  负例: 33200
  唯一用户数: 23125

预处理完成! 数据形状: (259958, 54)


In [3]:
from algo.classification import USER_FEATURES,compare_models
# 特征列（只使用用户特征）
feature_cols = [col for col in USER_FEATURES if col in data.columns]
print(f"\n使用的特征: {feature_cols}")

# 1. 多模型对比
results_df = compare_models(data, feature_cols)


使用的特征: ['age', 'gender', 'weight', 'height', 'level', 'under_weight', 'over_weight', 'blood_pressure', 'diabetes', 'anemia', 'osteoporosis', 'opioid_misuse', 'blood_urea_nitrogen', 'user_low_calorie', 'user_high_calorie', 'user_low_sugar', 'user_high_fiber', 'user_low_sodium', 'user_high_potassium', 'user_low_saturated_fat', 'user_low_cholesterol', 'user_low_protein', 'user_high_protein', 'bmi']

多种分类模型对比（用户特征 → 膳食匹配度）

训练 Logistic Regression...
  AUC: 0.7038, F1: 0.7940

训练 Random Forest...
  AUC: 0.7079, F1: 0.9155

训练 Gradient Boosting...
  AUC: 0.7278, F1: 0.9355

训练 XGBoost...
  AUC: 0.7266, F1: 0.9043

训练 LightGBM...
  AUC: 0.7264, F1: 0.9027

模型性能汇总
              Model  Accuracy  Precision   Recall       F1      AUC
  Gradient Boosting  0.882751   0.899219 0.974841 0.935504 0.727794
            XGBoost  0.837263   0.928298 0.881527 0.904308 0.726589
           LightGBM  0.834821   0.928086 0.878726 0.902732 0.726385
      Random Forest  0.854362   0.926642 0.904657 0.915517 0.7

In [4]:
from algo.classification import evaluate_ranking_capability,rule_baseline
# 2. 排序能力评估
ranking_metrics = evaluate_ranking_capability(data, feature_cols)

# 3. 简单规则基线
rule_metrics = rule_baseline(data, feature_cols)


模型排序能力评估

排序能力指标:
  AUC (ROC-AUC): 0.7266
  AP (PR-AUC): 0.9263

  结论: AUC=0.7266 > 0.7，模型具有较好的排序能力
  说明: 用户特征可以有效区分匹配和不匹配的膳食

简单规则基线

规则: user_low_sodium=1 → 匹配
  AUC: 0.6142

随机猜测 AUC: 0.5050
